# Western-U.S. HydroLAKES candidate discovery

Build and inspect the metadata-only candidate population used to plan a later regional threshold-transfer experiment.

## 1. Initialization

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'configs' / 'western_us.yaml').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'configs' / 'western_us.yaml').exists():
    raise FileNotFoundError('Run this notebook from the repository or notebooks directory.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from regional_analysis.config import load_config
from regional_analysis.discover_lakes import (
    build_western_state_collection,
    discover_candidate_inventory,
    initialize_earth_engine,
    validate_hydrolakes_schema,
    write_candidate_csv,
)

CONFIG_PATH = REPO_ROOT / 'configs' / 'western_us.yaml'
OUTPUT_PATH = REPO_ROOT / 'data' / 'regional' / 'western_hydrolakes_candidates.csv'
config = load_config(CONFIG_PATH)
ee, project = initialize_earth_engine()
print(f'Earth Engine project: {project}')

schema = validate_hydrolakes_schema(ee, config.earth_engine.hydrolakes_asset)
western_states = build_western_state_collection(
    ee, config.earth_engine.states_asset, config.region.states
)
print('Required HydroLAKES fields validated.')
print('Optional fields present:', list(schema.retained_optional_properties))
print('Optional fields absent:', list(schema.missing_optional_properties))

## 2. Candidate discovery

In [ ]:
result = discover_candidate_inventory(
    ee,
    config,
    schema=schema,
    state_collection=western_states,
)
write_candidate_csv(result.table, OUTPUT_PATH, config.region.states)
candidates = pd.read_csv(OUTPUT_PATH)
print(f'Wrote {len(candidates):,} candidates to {OUTPUT_PATH.relative_to(REPO_ROOT)}')

## 3. Population summary

In [ ]:
print(f'Total candidates: {len(candidates):,}')
display(candidates.groupby('state').size().rename('candidate_count').to_frame())
display(candidates.groupby('Lake_type', dropna=False).size().rename('candidate_count').to_frame())

missing_names = candidates['Lake_name'].isna() | candidates['Lake_name'].astype(str).str.strip().eq('')
display(pd.Series({
    'with_name': int((~missing_names).sum()),
    'without_name': int(missing_names.sum()),
}, name='count').to_frame())

display(candidates['Lake_area'].describe().rename('Lake_area_km2').to_frame())
display(candidates['Lake_area'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).rename('Lake_area_km2').to_frame())
display(pd.Series(result.summary['missing_optional_values'], name='missing_count').to_frame())
display(pd.DataFrame(result.summary['hydrolakes_sentinel_values']).T.fillna(0))
display(candidates.nlargest(40, 'Lake_area'))

## 4. Anchor-site QA

In [ ]:
for anchor_name, check in result.anchor_checks.items():
    print(f'{anchor_name}: {check.method}')
    print(check.note)
    display(check.records)

print(f'State-boundary centroid ambiguities: {len(result.boundary_ambiguities)}')
if not result.boundary_ambiguities.empty:
    display(result.boundary_ambiguities)

## 5. Lake-area distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

area = candidates['Lake_area'].dropna()
bins = np.geomspace(area.min(), area.max() * 1.001, 30)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(area, bins=bins, color='#3b82a0', edgecolor='white')
ax.set_xscale('log')
ax.set_xlabel('HydroLAKES Lake_area (km², log scale)')
ax.set_ylabel('Candidate count')
ax.set_title('Western-U.S. candidate lake-area distribution')
fig.tight_layout()

## 6. Interactive map

In [ ]:
import geemap

qa_map = geemap.Map(center=[39.5, -115.0], zoom=4)
state_boundaries = result.state_collection.style(
    color='5f6368', fillColor='00000000', width=2
)
candidate_polygons = result.candidate_collection.style(
    color='2166ac', fillColor='2166ac4d', width=1
)
qa_map.addLayer(state_boundaries, {}, 'Configured western states')
qa_map.addLayer(candidate_polygons, {}, 'HydroLAKES candidates')
qa_map

## 7. Notes

This inventory is the **candidate population**, not the final 20–30-lake experimental sample. The 25 km² setting is an experimental screening threshold for this regional-transfer analysis, not a scientific minimum lake size. Area alone should not determine experimental selection; later work should evaluate coverage, lake setting, reference-data suitability, and other design factors. The later analysis will evaluate **entire-lake threshold transfer**.